# Projet Machine Learning n°2 — Spotify

**Notebook complet :** analyse exploratoire, régression et classification.

Ce notebook regroupe les trois parties du projet. Les constantes et les fonctions de chargement/nettoyage sont définies ci-dessous : aucun import de `config.py` ni de `data.py` n'est nécessaire.

**Exécution :** utiliser **Kernel > Restart & Run All** dans Jupyter / VS Code. Les figures et résultats seront recalculés. Les bibliothèques nécessaires sont `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn` et `joblib`.

**Données :** le notebook ouvre `spotify_songs.csv` s'il est présent dans le dossier courant. Sinon, il récupère le CSV public d'origine (une connexion Internet est alors nécessaire).

## 0. Configuration, constantes et nettoyage des données

Ces éléments remplacent le contenu de `config.py` et de `data.py` pour rendre le notebook utilisable hors du dépôt du projet.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

# Paramètres communs
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_REG = "energy"
TARGET_CLF = "playlist_genre"

AUDIO_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness",
    "tempo", "key", "mode"
]

FEATURES_REG = [
    "danceability", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness",
    "tempo", "key", "mode", "duration_min", "release_year"
]

FEATURES_CLF = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness",
    "tempo", "key", "mode", "duration_min", "release_year"
]

# Jeu de données Spotify d'origine
URL_SPOTIFY = (
    "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/"
    "data/2020/2020-01-21/spotify_songs.csv"
)
DATA_FILE = Path("spotify_songs.csv")

def load_data():
    """Lit le CSV local lorsqu'il existe, sinon sa source publique."""
    if DATA_FILE.is_file():
        return pd.read_csv(DATA_FILE)
    return pd.read_csv(URL_SPOTIFY)

def clean_data(df):
    """Nettoyage commun aux trois parties du projet."""
    df_clean = df.copy()
    # Garder la ligne la plus populaire en cas de doublon
    df_clean = df_clean.sort_values(
        "track_popularity", ascending=False, kind="stable"
    )
    df_clean = df_clean.drop_duplicates(subset=["track_id"], keep="first")
    df_clean = df_clean.drop_duplicates(
        subset=["track_artist", "track_name", "track_album_name", "duration_ms"]
        + AUDIO_FEATURES,
        keep="first"
    )
    df_clean["release_year"] = pd.to_datetime(
        df_clean["track_album_release_date"], format="mixed"
    ).dt.year
    df_clean["duration_min"] = df_clean["duration_ms"] / 60000
    df_clean["playlist_genre"] = df_clean["playlist_genre"].astype("category")
    return df_clean

df = load_data()
df_clean = clean_data(df)
print(f"Données brutes : {len(df):,} lignes")
print(f"Après nettoyage : {len(df_clean):,} morceaux")
print(f"Doublons track_id : {df_clean['track_id'].duplicated().sum()}")

---
# Partie 1 — Analyse exploratoire (EDA)

Étude du jeu de données, vérification du nettoyage et observation des caractéristiques musicales.

In [ ]:
# 1. Structure : dimensions, types et valeurs manquantes
print("Dimensions :", df.shape)
df.info()
print(df.isnull().sum())
display(df.describe())

In [ ]:
# 2. Doublons
print("Nombre de lignes :", len(df))
print("Morceaux uniques :", df["track_id"].nunique())
nb_genres = df.groupby("track_id")["playlist_genre"].nunique()
print("Morceaux avec plusieurs genres :", (nb_genres > 1).sum())

Le dataset initial contient 32 833 lignes, mais seulement 28 356 morceaux uniques identifiés par leur `track_id`. Cette différence s'explique par la présence de morceaux dans plusieurs playlists. Parmi ces morceaux, 1 687 sont associés à plusieurs genres musicaux. Lors du nettoyage, nous conservons une seule occurrence par identifiant, mais cela implique de ne retenir qu'un seul genre pour certains morceaux.

In [ ]:
# 3. Changements de types et nouvelles variables
print(df_clean[["release_year", "duration_min", "playlist_genre"]].head())
print(df_clean.dtypes)
print("Doublons restants :", df_clean["track_id"].duplicated().sum())

In [ ]:
# 4.1 Distribution des caractéristiques audio

df_clean[AUDIO_FEATURES].hist(
    figsize=(18, 14), bins=30, layout=(4, 3), edgecolor="white"
)
plt.suptitle("Distribution des caractéristiques audio", fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# 4.2 Distribution de la popularité
plt.figure(figsize=(12, 5))
sns.histplot(data=df_clean, x="track_popularity", bins=30, edgecolor="white")
plt.title("Distribution de la popularité des morceaux", fontsize=15)
plt.xlabel("Popularité (0 à 100)")
plt.ylabel("Nombre de morceaux")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

# 4.3 Distribution de la durée
plt.figure(figsize=(12, 5))
sns.histplot(data=df_clean, x="duration_min", bins=40, edgecolor="white")
plt.title("Distribution de la durée des morceaux", fontsize=15)
plt.xlabel("Durée (minutes)")
plt.ylabel("Nombre de morceaux")
plt.xlim(0, 15)
plt.tight_layout()
plt.show()

# 4.4 Nombre de morceaux par genre
ordre_genres = df_clean["playlist_genre"].value_counts().index
plt.figure(figsize=(11, 6))
ax = sns.countplot(data=df_clean, x="playlist_genre", order=ordre_genres)
plt.title("Nombre de morceaux par genre musical", fontsize=15)
plt.xlabel("Genre musical")
plt.ylabel("Nombre de morceaux")
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.ylim(0, ax.get_ylim()[1] * 1.1)
plt.tight_layout()
plt.show()
print("Nombre de morceaux par genre :")
print(df_clean["playlist_genre"].value_counts())

# 4.5 Nombre de morceaux par sous-genre
ordre_sous_genres = df_clean["playlist_subgenre"].value_counts().index
plt.figure(figsize=(13, 10))
ax = sns.countplot(data=df_clean, y="playlist_subgenre", order=ordre_sous_genres)
plt.title("Nombre de morceaux par sous-genre musical", fontsize=15)
plt.xlabel("Nombre de morceaux")
plt.ylabel("Sous-genre")
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.xlim(0, ax.get_xlim()[1] * 1.12)
plt.tight_layout()
plt.show()

### Observations sur les distributions

**Caractéristiques audio :** les morceaux Spotify présentent généralement une énergie (`energy`) et une capacité à danser (`danceability`) assez élevées. Les valeurs de `speechiness`, `acousticness` et `instrumentalness` sont souvent proches de zéro. Le tempo se situe principalement entre 90 et 140 BPM. La variable `valence` est assez bien répartie entre 0 et 1.

**Popularité :** les scores se concentrent principalement entre 30 et 70. On observe également de nombreux morceaux ayant une popularité égale à zéro, sans pouvoir en déterminer la cause avec ce seul graphique.

Nous avons identifié plusieurs morceaux avec des identifiants Spotify différents mais des informations similaires. Nous avons donc adapté le nettoyage pour conserver la version la plus populaire lorsque les caractéristiques audio sont identiques. Après nettoyage, 28 068 morceaux restent sur 32 833 lignes. Les versions présentant des caractéristiques audio différentes sont conservées.

In [ ]:
# 5.1 Matrice de corrélation
correlation = df_clean[AUDIO_FEATURES].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Corrélation entre les caractéristiques audio", fontsize=15)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
print("Corrélations avec energy :")
print(correlation["energy"].sort_values(ascending=False))

**Loudness (+0,68) :** plus un morceau est fort en volume sonore, plus son énergie a tendance à être élevée.

**Acousticness (−0,55) :** les morceaux ayant une forte composante acoustique ont généralement une énergie plus faible.

In [ ]:
# 5.2 Boxplots des caractéristiques audio par genre
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df_clean, x="playlist_genre", y="energy", ax=axes[0])
axes[0].set_title("Énergie selon le genre")
axes[0].set_xlabel("Genre musical")
axes[0].set_ylabel("Énergie")
sns.boxplot(data=df_clean, x="playlist_genre", y="danceability", ax=axes[1])
axes[1].set_title("Danceability selon le genre")
axes[1].set_xlabel("Genre musical")
axes[1].set_ylabel("Danceability")
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Popularité moyenne par genre
popularite_genre = (
    df_clean.groupby("playlist_genre", observed=True)["track_popularity"]
    .mean().sort_values(ascending=False)
)
plt.figure(figsize=(11, 6))
ax = sns.barplot(x=popularite_genre.index, y=popularite_genre.values)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)
plt.title("Popularité moyenne selon le genre", fontsize=15)
plt.xlabel("Genre musical")
plt.ylabel("Popularité moyenne (0 à 100)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()
print(popularite_genre.round(2))

In [ ]:
# 5.4 Popularité selon l'année de sortie
popularite_annee = df_clean.groupby("release_year").agg(
    popularite_moyenne=("track_popularity", "mean"),
    nombre_morceaux=("track_id", "size")
)
popularite_annee = popularite_annee[popularite_annee["nombre_morceaux"] >= 30]
plt.figure(figsize=(13, 5))
sns.lineplot(data=popularite_annee, x=popularite_annee.index, y="popularite_moyenne", marker="o")
plt.title("Popularité moyenne selon l'année de sortie", fontsize=15)
plt.xlabel("Année de sortie")
plt.ylabel("Popularité moyenne")
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 Relation Loudness / Energy selon le genre
# Échantillon pour rendre le graphique plus lisible
echantillon = df_clean.sample(n=min(3000, len(df_clean)), random_state=RANDOM_STATE)
plt.figure(figsize=(12, 7))
sns.scatterplot(data=echantillon, x="loudness", y="energy", hue="playlist_genre", alpha=0.5, s=30)
plt.title("Relation entre Loudness et Energy par genre", fontsize=15)
plt.xlabel("Loudness (dB)")
plt.ylabel("Energy (0 à 1)")
plt.legend(title="Genre musical", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# 6.1 Top 10 des artistes les plus représentés
top_artistes = df_clean["track_artist"].value_counts().head(10)
plt.figure(figsize=(12, 6))
ax = sns.barplot(x=top_artistes.values, y=top_artistes.index)
plt.title("Top 10 des artistes les plus présents", fontsize=15)
plt.xlabel("Nombre de morceaux")
plt.ylabel("Artiste")
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.xlim(0, top_artistes.max() * 1.15)
plt.tight_layout()
plt.show()
print("Top 10 des artistes :")
print(top_artistes)

In [ ]:
# 6.2 Top 10 des morceaux les plus populaires
top_morceaux = (
    df_clean.sort_values("track_popularity", ascending=False, kind="stable")
    .drop_duplicates(subset=["track_artist", "track_name"])
    .head(10).copy()
)
top_morceaux["morceau"] = (
    top_morceaux["track_name"].str.slice(0, 35)
    + " - " + top_morceaux["track_artist"].str.slice(0, 20)
)
plt.figure(figsize=(13, 7))
ax = sns.barplot(data=top_morceaux, x="track_popularity", y="morceau")
plt.title("Top 10 des morceaux les plus populaires", fontsize=15)
plt.xlabel("Score de popularité")
plt.ylabel("Morceau")
plt.xlim(0, 110)
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.tight_layout()
plt.show()
print("Top 10 des morceaux les plus populaires :")
display(top_morceaux[["track_name", "track_artist", "track_album_name", "release_year", "track_popularity"]])

## 7. Conclusions de l'analyse exploratoire

**1. Présence de doublons :** le dataset contient initialement 32 833 lignes. Après nettoyage, nous conservons 28 068 morceaux. Nous avons supprimé les doublons en conservant les versions les plus populaires lorsque les identifiants ou les caractéristiques audio permettent de les considérer comme identiques.

**2. Certains morceaux appartiennent à plusieurs genres :** nous avons identifié 1 687 morceaux associés à plusieurs genres musicaux dans le dataset initial. Cela complique la classification car un même morceau peut figurer dans différentes playlists.

**3. Des morceaux généralement énergétiques et dansants :** les distributions montrent que les morceaux possèdent souvent des valeurs assez élevées d'`energy` et de `danceability`. À l'inverse, `speechiness`, `acousticness` et `instrumentalness` sont généralement faibles.

**4. Une corrélation importante entre volume et énergie :** `loudness` présente une corrélation positive de +0,68 avec `energy`.

**5. Une relation négative entre acousticness et energy :** `acousticness` présente une corrélation de −0,55 avec `energy`. Cette variable pourrait donc être utile pour la régression.

---
# Partie 2 — Régression

Comparaison de modèles linéaires et Random Forest. La version récente du notebook étudie **l'énergie**, avec et sans genre, puis **la popularité**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")

# 1. Chargement, analyse des proportions et stratégie
# Les données déjà chargées et nettoyées au début sont réutilisées.
total = len(df_clean)

df_base = df_clean[
    (df_clean['duration_min'] >= 2.3) & (df_clean['duration_min'] <= 5.0) &
    (df_clean['loudness'] >= -20.0) & (df_clean['tempo'] > 0) &
    (df_clean['speechiness'] < 0.5)
].copy()

NUMERIC_FEATURES = [
    'danceability', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'loudness', 'tempo', 'key', 'mode',
    'duration_min', 'release_year'
]

def get_metrics(name, model, X_te, y_te):
    pred = model.predict(X_te)
    return {
        'Modèle': name,
        'R²': round(r2_score(y_te, pred), 3),
        'MAE': round(mean_absolute_error(y_te, pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_te, pred)), 3)
    }

print(f"Données nettoyées : {total} morceaux")
print(f"Après les filtres de régression : {len(df_base)} morceaux")

In [ ]:
# PARTIE A : PRÉDICTION DE L'ÉNERGIE (AVEC ET SANS GENRE)
print("\n" + "="*50)
print(" PARTIE A : PRÉDICTION DE L'ÉNERGIE (AVEC & SANS GENRE)")
print("="*50)

TARGET_ENR = 'energy'

# 1. Version SANS le genre
X_train_sans, X_test_sans, y_train_sans, y_test_sans = train_test_split(
    df_base[NUMERIC_FEATURES], df_base[TARGET_ENR], test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)
model_lr_sans = LinearRegression().fit(X_train_sans, y_train_sans)
model_rf_sans = RandomForestRegressor(
    random_state=RANDOM_STATE, n_estimators=100
).fit(X_train_sans, y_train_sans)

# 2. Version AVEC le genre (One-Hot Encoding)
df_encoded = pd.get_dummies(
    df_base, columns=['playlist_genre'], drop_first=False, dtype=int
)
GENRE_FEATURES = [col for col in df_encoded.columns if col.startswith('playlist_genre_')]
FEATURES_ENERGY_AVEC = NUMERIC_FEATURES + GENRE_FEATURES

X_train_avec, X_test_avec, y_train_avec, y_test_avec = train_test_split(
    df_encoded[FEATURES_ENERGY_AVEC], df_encoded[TARGET_ENR],
    test_size=TEST_SIZE, random_state=RANDOM_STATE
)
model_lr_avec = LinearRegression().fit(X_train_avec, y_train_avec)
model_rf_avec = RandomForestRegressor(
    random_state=RANDOM_STATE, n_estimators=100
).fit(X_train_avec, y_train_avec)

results_nrj = pd.DataFrame([
    get_metrics("Linéaire (Sans Genre)", model_lr_sans, X_test_sans, y_test_sans),
    get_metrics("Linéaire (Avec Genre)", model_lr_avec, X_test_avec, y_test_avec),
    get_metrics("Random Forest (Sans Genre)", model_rf_sans, X_test_sans, y_test_sans),
    get_metrics("Random Forest (Avec Genre)", model_rf_avec, X_test_avec, y_test_avec),
])
display(results_nrj)

In [ ]:
# PARTIE B : PRÉDICTION DE LA POPULARITÉ (SANS GENRE)
print("\n" + "="*50)
print(" PARTIE B : PRÉDICTION DE LA POPULARITÉ")
print("="*50)

TARGET_POP = 'track_popularity'
X_train_pop, X_test_pop, y_train_pop, y_test_pop = train_test_split(
    df_base[NUMERIC_FEATURES], df_base[TARGET_POP],
    test_size=TEST_SIZE, random_state=RANDOM_STATE
)
model_pop_lr = LinearRegression().fit(X_train_pop, y_train_pop)
model_pop_rf = RandomForestRegressor(
    random_state=RANDOM_STATE, n_estimators=100, max_depth=15
).fit(X_train_pop, y_train_pop)
results_pop = pd.DataFrame([
    get_metrics("Linéaire (Popularité)", model_pop_lr, X_test_pop, y_test_pop),
    get_metrics("Random Forest (Popularité)", model_pop_rf, X_test_pop, y_test_pop)
])
display(results_pop)

---
# Partie 3 — Classification : prédire le genre musical (`playlist_genre`)

L'objectif est de prédire le genre musical d'un morceau parmi **6 classes** (`edm`, `latin`, `pop`, `r&b`, `rap`, `rock`) à partir des caractéristiques audio, de la durée et de la popularité.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")

# 3.1 Préparation et train/test split
features_clf_propre = FEATURES_CLF + ['track_popularity']
X_clf = df_clean[features_clf_propre]
y_clf = df_clean[TARGET_CLF]

print(f"Dimensions de X_clf : {X_clf.shape}")
print(f"Dimensions de y_clf : {y_clf.shape}")

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_clf
)
print(f"Taille du train : {X_clf_train.shape[0]} lignes | Taille du test : {X_clf_test.shape[0]} lignes")

### 3.2 Modèle naïf : baseline

Un modèle qui répond uniformément au hasard parmi six genres aurait une accuracy moyenne de **1/6, soit 16,67 %**. Le modèle naïf prédit toujours le genre majoritaire dans le jeu d'entraînement. Ce résultat constitue un point de comparaison.

In [ ]:
majority_class = y_clf_train.mode()[0]
y_naive_pred = [majority_class] * len(y_clf_test)
naive_accuracy = accuracy_score(y_clf_test, y_naive_pred)
print(f"Classe majoritaire prédite : {majority_class}")
print(f"Accuracy du modèle naïf (score plancher à battre) : {naive_accuracy:.4f}")

### 3.3 et 3.4 Régression logistique et KNN

**Régression logistique :** standardisation avec `StandardScaler` intégré à un pipeline pour éviter les fuites de données.

**KNN :** comparaison de plusieurs nombres de voisins (`k`) pour observer le surapprentissage : les petits `k` peuvent donner un score élevé sur l'entraînement mais se généraliser moins bien.

In [ ]:
# Régression logistique
model_clf_log = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
model_clf_log.fit(X_clf_train, y_clf_train)
y_pred_log = model_clf_log.predict(X_clf_test)
log_accuracy = accuracy_score(y_clf_test, y_pred_log)
log_f1_macro = f1_score(y_clf_test, y_pred_log, average='macro')

# KNN : différentes valeurs de k
k_values = [3, 5, 11, 21]
knn_results = []
best_k_acc = 0
for k in k_values:
    knn_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    knn_pipeline.fit(X_clf_train, y_clf_train)
    acc_train = accuracy_score(y_clf_train, knn_pipeline.predict(X_clf_train))
    acc_test = accuracy_score(y_clf_test, knn_pipeline.predict(X_clf_test))
    knn_results.append({'k': k, 'Train Acc': acc_train, 'Test Acc': acc_test})
    if acc_test > best_k_acc:
        best_k_acc = acc_test

df_knn_results = pd.DataFrame(knn_results)
print(f"Régression Logistique - Accuracy : {log_accuracy:.4f}")
print("\nRésultats du KNN (observation de l'overfitting) :")
display(df_knn_results)

### 3.5 Random Forest et importance des variables

Le **Random Forest** combine plusieurs arbres de décision et peut apprendre des relations non linéaires entre les variables.

In [ ]:
model_clf_rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
model_clf_rf.fit(X_clf_train, y_clf_train)
y_pred_rf = model_clf_rf.predict(X_clf_test)
rf_accuracy = accuracy_score(y_clf_test, y_pred_rf)
rf_f1_macro = f1_score(y_clf_test, y_pred_rf, average='macro')
print(f"Random Forest - Accuracy : {rf_accuracy:.4f}")
print(f"Random Forest - F1-score (macro) : {rf_f1_macro:.4f}\n")

importances = model_clf_rf.feature_importances_
df_importance = pd.DataFrame({'Feature': X_clf.columns, 'Importance': importances})
df_importance = df_importance.sort_values(by='Importance', ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=df_importance, color='#1DB954')
plt.title("Importance des variables explicatives (Random Forest)")
plt.tight_layout()
plt.show()

### 3.6 Évaluation et matrice de confusion

Les résultats du notebook d'origine indiquaient que le Rock et l'EDM étaient généralement les genres les mieux reconnus. La Pop et le R&B étaient davantage confondus. Les scores ci-dessous sont recalculés avec le nettoyage courant.

In [ ]:
print(classification_report(y_clf_test, y_pred_rf))

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_clf_test, y_pred_rf, labels=model_clf_rf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model_clf_rf.classes_)
disp.plot(cmap=plt.cm.Blues, xticks_rotation='vertical', ax=plt.gca())
plt.title("Matrice de Confusion 6x6 — Random Forest")
plt.tight_layout()
plt.show()

print("\n--- Analyse des probabilités sur 3 exemples (predict_proba) ---")
probas = model_clf_rf.predict_proba(X_clf_test[:3])
for i, p in enumerate(probas):
    print(f"Morceau {i + 1} — Véritable genre : {y_clf_test.iloc[i]}")
    for genre, prob in zip(model_clf_rf.classes_, p):
        print(f"   -> {genre}: {prob * 100:.2f}%")

### 3.7 Tableau comparatif final

Les performances des modèles sont comparées sur le jeu de test. Le Random Forest est ensuite exporté dans un sous-dossier `models/` créé localement.

In [ ]:
results_clf = pd.DataFrame([
    {"Modèle": "Modèle Naïf (Baseline)", "Accuracy": naive_accuracy, "F1-score (macro)": "N/A"},
    {"Modèle": "Régression Logistique", "Accuracy": log_accuracy, "F1-score (macro)": log_f1_macro},
    {"Modèle": "KNN Optimisé", "Accuracy": best_k_acc, "F1-score (macro)": "N/A"},
    {"Modèle": "Random Forest", "Accuracy": rf_accuracy, "F1-score (macro)": rf_f1_macro}
])
print("--- Tableau Comparatif des Performances ---")
display(results_clf)

# Export local facultatif pour une éventuelle application Streamlit
EXPORTER_MODELE = False
if EXPORTER_MODELE:
    Path("models").mkdir(exist_ok=True)
    joblib.dump(model_clf_rf, "models/classification.joblib")
    print("Modèle sauvegardé dans models/classification.joblib")